[List of Italian provinces by life expectancy](https://en.wikipedia.org/wiki/List_of_Italian_provinces_by_life_expectancy) <i>([alternative](https://en.wikipedia.org/wiki/User:Lady3mlnm/List_of_Italian_provinces_by_life_expectancy_(alternative)))</i> / [Продолжительность жизни в провинциях Италии](https://ru.wikipedia.org/wiki/Продолжительность_жизни_в_провинциях_Италии)<br />
[Administrative divisions of Italy](https://en.wikipedia.org/wiki/Italy#Administrative_divisions) / 
[Административное деление Италии](https://ru.wikipedia.org/wiki/Административное_деление_Италии)<br />
source of data: [ISTAT](https://demo.istat.it/tavole/?t=indicatori)

In [2]:
import pandas as pd
import math
import re

import sys
sys.path.append("..")
import mal_moduls_private.mal_total as mal

In [3]:
pd.options.display.min_rows = 4
pd.options.display.max_rows = 50
pd.options.display.max_columns = 30

In [4]:
# load stats about longevity
df = pd.read_excel('data/Indicatori_demografici -2024.xls', sheet_name='speranza_di_vita', skiprows=4, header=None, index_col=0)
df.index.name=''

df.dropna(how='all', inplace=True)

df

,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,...,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138
,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Torino,77.4,17.0,83.1,20.9,80.2,19.0,77.2,16.7,82.8,20.6,79.9,18.7,78.2,17.5,83.7,...,21.9,82.8,20.5,81.5,19.7,85.4,22.5,83.4,21.1,81.9,20.0,85.9,22.9,83.9,21.4
Vercelli,76.3,16.6,81.3,20.0,78.7,18.3,76.1,16.3,81.4,20.1,78.7,18.3,76.2,16.4,83.1,...,21.5,81.7,19.9,79.8,18.8,84.4,22.2,82.0,20.5,80.4,19.2,84.8,22.5,82.6,20.9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ISOLE,76.8,16.8,82.1,20.1,79.4,18.5,76.8,16.7,81.9,19.8,79.3,18.3,77.7,17.4,82.8,...,21.2,81.5,19.7,79.8,18.7,84.1,21.5,81.9,20.1,80.5,19.3,84.7,22.2,82.5,20.8
ITALIA,77.2,16.9,83.0,20.8,80.0,18.9,77.2,16.8,82.8,20.5,80.0,18.7,77.9,17.3,83.6,...,21.9,82.6,20.4,81.0,19.4,85.1,22.3,83.0,20.9,81.5,19.9,85.6,22.7,83.5,21.3


In [5]:
ls_cols = []
for year in list(range(2002, 2025)):
    ls_cols.extend([f'm_{year}', f'm65_{year}', f'f_{year}', f'f65_{year}', f'{year}', f't65_{year}'])

df.columns = ls_cols
df

,m_2002,m65_2002,f_2002,f65_2002,2002,t65_2002,m_2003,m65_2003,f_2003,f65_2003,2003,t65_2003,m_2004,m65_2004,f_2004,...,f65_2022,2022,t65_2022,m_2023,m65_2023,f_2023,f65_2023,2023,t65_2023,m_2024,m65_2024,f_2024,f65_2024,2024,t65_2024
,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Torino,77.4,17.0,83.1,20.9,80.2,19.0,77.2,16.7,82.8,20.6,79.9,18.7,78.2,17.5,83.7,...,21.9,82.8,20.5,81.5,19.7,85.4,22.5,83.4,21.1,81.9,20.0,85.9,22.9,83.9,21.4
Vercelli,76.3,16.6,81.3,20.0,78.7,18.3,76.1,16.3,81.4,20.1,78.7,18.3,76.2,16.4,83.1,...,21.5,81.7,19.9,79.8,18.8,84.4,22.2,82.0,20.5,80.4,19.2,84.8,22.5,82.6,20.9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ISOLE,76.8,16.8,82.1,20.1,79.4,18.5,76.8,16.7,81.9,19.8,79.3,18.3,77.7,17.4,82.8,...,21.2,81.5,19.7,79.8,18.7,84.1,21.5,81.9,20.1,80.5,19.3,84.7,22.2,82.5,20.8
ITALIA,77.2,16.9,83.0,20.8,80.0,18.9,77.2,16.8,82.8,20.5,80.0,18.7,77.9,17.3,83.6,...,21.9,82.6,20.4,81.0,19.4,85.1,22.3,83.0,20.9,81.5,19.9,85.6,22.7,83.5,21.3


<br />
<br />

In [7]:
def sort_df(df, by=['2024', 'm_2024', 'f_2024'], ascending=False):
    df.sort_values(by=by, ascending=ascending, inplace=True)
    indexes = df.index.to_list()
    ls_change_order = ['ITALIA']
    indexes = list(el for el in indexes if el not in ls_change_order)
    indexes = ls_change_order + indexes
    return df.reindex(indexes)

In [8]:
def extend_df(df, prec=1):
    df.insert(loc=1,  column='2014→2019', value=(df['2019']-df['2014']).round(prec))
    df.insert(loc=3,  column='2019→2020', value=(df['2020']-df['2019']).round(prec))
    df.insert(loc=5,  column='2020→2021', value=(df['2021']-df['2020']).round(prec))
    df.insert(loc=7,  column='2021→2022', value=(df['2022']-df['2021']).round(prec))
    df.insert(loc=9,  column='2022→2023', value=(df['2023']-df['2022']).round(prec))
    df.insert(loc=11,  column='2023→2024', value=(df['2024']-df['2023']).round(prec))
    df.insert(loc=13, column='2019→2024', value=(df['2024']-df['2019']).round(prec))
    df.insert(loc=14, column='2014→2024', value=(df['2024']-df['2014']).round(prec))
    df['f-m_2014'] = (df['f_2014']-df['m_2014']).round(prec)
    df['f-m_2019'] = (df['f_2019']-df['m_2019']).round(prec)
    df['f-m_2024'] = (df['f_2024']-df['m_2024']).round(prec)

<br />
<br />
<br />

---
<h4>Statistics for regions</h4>

In [10]:
regions = ["Piemonte", "Valle d'Aosta", "Lombardia", "Trentino-Alto Adige", "Veneto", "Friuli-Venezia Giulia",
           "Liguria", "Emilia-Romagna", "Toscana", "Umbria", "Marche", "Lazio", "Abruzzo", "Molise", "Campania",
           "Puglia", "Basilicata", "Calabria", "Sicilia", "Sardegna"]

macro_regions = ["NORD", "NORD-OVEST", "NORD-EST", "CENTRO", "MEZZOGIORNO", "SUD", "ISOLE"]

cols_to_table = ['2014', '2019', '2020', '2021', '2022', '2023', '2024',
                 'm_2014', 'm_2019', 'm_2024', 'f_2014', 'f_2019', 'f_2024']

df_regions = df.loc[['ITALIA'] + regions, cols_to_table]

df_regions = sort_df(df_regions)

df_regions.head(3)

,2014,2019,2020,2021,2022,2023,2024,m_2014,m_2019,m_2024,f_2014,f_2019,f_2024
,,,,,,,,,,,,,
ITALIA,82.6,83.2,82.1,82.5,82.6,83.0,83.5,80.3,81.1,81.5,85.0,85.4,85.6
Trentino-Alto Adige,83.5,84.2,82.8,83.6,83.8,84.4,84.7,81.2,81.9,82.8,85.9,86.5,86.8
Veneto,83.2,83.8,82.9,83.3,83.4,83.8,84.3,80.7,81.7,82.4,85.8,86.1,86.4


In [11]:
extend_df(df_regions)  # CAUTION: after this operation given to the function DataFrame becomes extended, operation of assignment is redundant

df_regions

,2014,2014→2019,2019,2019→2020,2020,2020→2021,2021,2021→2022,2022,2022→2023,2023,2023→2024,2024,2019→2024,2014→2024,m_2014,m_2019,m_2024,f_2014,f_2019,f_2024,f-m_2014,f-m_2019,f-m_2024
,,,,,,,,,,,,,,,,,,,,,,,,
ITALIA,82.6,0.6,83.2,-1.1,82.1,0.4,82.5,0.1,82.6,0.4,83.0,0.5,83.5,0.3,0.9,80.3,81.1,81.5,85.0,85.4,85.6,4.7,4.3,4.1
Trentino-Alto Adige,83.5,0.7,84.2,-1.4,82.8,0.8,83.6,0.2,83.8,0.6,84.4,0.3,84.7,0.5,1.2,81.2,81.9,82.8,85.9,86.5,86.8,4.7,4.6,4.0
Veneto,83.2,0.6,83.8,-0.9,82.9,0.4,83.3,0.1,83.4,0.4,83.8,0.5,84.3,0.5,1.1,80.7,81.7,82.4,85.8,86.1,86.4,5.1,4.4,4.0
Marche,83.3,0.7,84.0,-1.0,83.0,0.1,83.1,0.2,83.3,0.6,83.9,0.4,84.3,0.3,1.0,81.1,81.9,82.4,85.7,86.1,86.2,4.6,4.2,3.8
Emilia-Romagna,83.2,0.4,83.6,-1.1,82.5,0.5,83.0,0.3,83.3,0.3,83.6,0.5,84.1,0.5,0.9,81.0,81.7,82.4,85.4,85.7,85.9,4.4,4.0,3.5
Toscana,83.2,0.4,83.6,-0.5,83.1,0.1,83.2,0.1,83.3,0.5,83.8,0.3,84.1,0.5,0.9,81.1,81.7,82.3,85.5,85.7,85.9,4.4,4.0,3.6
Lombardia,83.2,0.4,83.6,-2.2,81.4,1.7,83.1,0.1,83.2,0.6,83.8,0.3,84.1,0.5,0.9,80.9,81.5,82.2,85.6,85.9,86.1,4.7,4.4,3.9
Umbria,83.2,0.8,84.0,-0.6,83.4,-0.2,83.2,0.1,83.3,0.4,83.7,0.3,84.0,0.0,0.8,81.0,82.0,82.3,85.6,86.2,85.9,4.6,4.2,3.6
Friuli-Venezia Giulia,82.7,0.8,83.5,-0.9,82.6,-0.3,82.3,0.7,83.0,0.4,83.4,0.5,83.9,0.4,1.2,80.3,81.3,81.8,85.2,85.8,86.1,4.9,4.5,4.3


<br />
<br />

In [13]:
mal.min_and_max_values(
    df_regions[['2014', '2014→2019', '2019', '2019→2024', '2024', '2014→2024']],
    row_center='ITALIA',  max_lng=13)

Number of records: 21


,2014,2014→2019,2019,2019→2024,2024,2014→2024
max,83.5 -Trentino-Alt…,0.9 -Lazio,84.2 -Trentino-Alt…,0.7 -Basilicata,84.7 -Trentino-Alt…,1.2 -Trentino-Alt…
max_2,83.3 -Marche,0.8 -Umbria,84.0 -Marche,0.6 -Piemonte,84.3 -Veneto,1.2 -Friuli-Venez…
max_3,83.2 -Veneto,0.8 -Friuli-Venez…,84.0 -Umbria,0.5 -Trentino-Alt…,84.3 -Marche,1.1 -Veneto
ITALIA,– 82.6 –,– 0.6 –,– 83.2 –,– 0.3 –,– 83.5 –,– 0.9 –
min_3,81.9 -Calabria,0.4 -Emilia-Romag…,82.4 -Calabria,0.0 -Sardegna,82.5 -Molise,0.7 -Puglia
min_2,81.6 -Sicilia,0.3 -Piemonte,82.0 -Sicilia,0.0 -Umbria,82.4 -Sicilia,0.5 -Sardegna
min,80.9 -Campania,0.1 -Basilicata,81.6 -Campania,-0.5 -Molise,81.8 -Campania,0.2 -Molise


In [14]:
mal.min_and_max_values(
    df_regions[['m_2014', 'm_2019', 'm_2024', 'f_2014', 'f_2019', 'f_2024', 'f-m_2014', 'f-m_2019', 'f-m_2024']],
    row_center='ITALIA',  max_lng=13)

Number of records: 21


,m_2014,m_2019,m_2024,f_2014,f_2019,f_2024,f-m_2014,f-m_2019,f-m_2024
max,81.2 -Trentino-Alt…,82.0 -Umbria,82.8 -Trentino-Alt…,85.9 -Trentino-Alt…,86.5 -Trentino-Alt…,86.8 -Trentino-Alt…,5.4 -Sardegna,5.7 -Valle d'Aosta,5.1 -Sardegna
max_2,81.1 -Marche,81.9 -Trentino-Alt…,82.4 -Veneto,85.8 -Veneto,86.2 -Umbria,86.4 -Veneto,5.4 -Molise,5.5 -Sardegna,5.0 -Molise
max_3,81.1 -Toscana,81.9 -Marche,82.4 -Marche,85.7 -Marche,86.1 -Veneto,86.2 -Marche,5.1 -Veneto,5.3 -Molise,4.5 -Calabria
ITALIA,– 80.3 –,– 81.1 –,– 81.5 –,– 85.0 –,– 85.4 –,– 85.6 –,– 4.7 –,– 4.3 –,– 4.1 –
min_3,79.5 -Sicilia,80.0 -Sicilia,80.4 -Calabria,84.4 -Calabria,84.7 -Basilicata,84.7 -Valle d'Aosta,4.4 -Toscana,4.0 -Puglia,3.6 -Umbria
min_2,79.5 -Calabria,79.9 -Valle d'Aosta,80.0 -Molise,83.8 -Sicilia,84.0 -Sicilia,84.4 -Sicilia,4.4 -Emilia-Romag…,4.0 -Toscana,3.6 -Toscana
min,78.6 -Campania,79.6 -Campania,79.8 -Campania,83.3 -Campania,83.8 -Campania,84.0 -Campania,4.3 -Sicilia,4.0 -Emilia-Romag…,3.5 -Emilia-Romag…


<br />
<br />

In [16]:
# for name in sorted(df_regions.index.to_list()):
#     print(f'    "{name}"   : {{"it": ("", ""), "en": ("", ""), "ru": ("", "")}},')

In [17]:
dd_replacement_regions = {
    "ITALIA"   : {"it": ("Italia", ""), "en": ("Italy on average", ""), "ru": ("Италия в среднем", "")},
    "Abruzzo"   : {"it": ("Abruzzo", "Abruzzo"), "en": ("Abruzzo", "Abruzzo"), "ru": ("Абру́ццо", "Абруцци")},
    "Basilicata"   : {"it": ("Basilicata", "Basilicata"), "en": ("Basilicata", "Basilicata"), "ru": ("Базилика́та", "Базиликата")},
    "Calabria"   : {"it": ("Calabria", "Calabria"), "en": ("Calabria", "Calabria"), "ru": ("Кала́брия", "Калабрия")},
    "Campania"   : {"it": ("Campania", "Campania"), "en": ("Campania", "Campania"), "ru": ("Кампа́ния", "Кампания (Италия)")},
    "Emilia-Romagna"   : {"it": ("Emilia-Romagna", "Emilia-Romagna"), "en": ("Emilia-Romagna", "Emilia-Romagna"), "ru": ("Эми́лия-Рома́нья", "Эмилия-Романья")},
    "Friuli-Venezia Giulia"   : {"it": ("Friuli-Venezia Giulia", "Friuli-Venezia Giulia"), "en": ("Friuli-Venezia Giulia", "Friuli-Venezia Giulia"), "ru": ("Фриу́ли-Вене́ция-Джу́лия", "Фриули-Венеция-Джулия")},
    "Lazio"   : {"it": ("Lazio", "Lazio"), "en": ("Lazio", "Lazio"), "ru": ("Ла́цио", "Лацио")},
    "Liguria"   : {"it": ("Liguria", "Liguria"), "en": ("Liguria", "Liguria"), "ru": ("Лигу́рия", "Лигурия")},
    "Lombardia"   : {"it": ("Lombardia", "Lombardia"), "en": ("Lombardy", "Lombardy"), "ru": ("Ломба́рдия", "Ломбардия")},
    "Marche"   : {"it": ("Marche", "Marche"), "en": ("Marche", "Marche"), "ru": ("Ма́рке", "Марке")},
    "Molise"   : {"it": ("Molise", "Molise"), "en": ("Molise", "Molise"), "ru": ("Моли́зе", "Молизе")},
    "Piemonte"   : {"it": ("Piemonte", "Piemonte"), "en": ("Piedmont", "Piedmont"), "ru": ("Пьемо́нт", "Пьемонт")},
    "Puglia"   : {"it": ("Puglia", "Puglia"), "en": ("Apulia (Puglia)", "Apulia"), "ru": ("Апу́лия ", "Апулия")},
    "Sardegna"   : {"it": ("Sardegna", "Sardegna"), "en": ("Sardinia", "Sardinia"), "ru": ("Сарди́ния", "Сардиния")},
    "Sicilia"   : {"it": ("Sicilia", "Sicilia"), "en": ("Sicily", "Sicily"), "ru": ("Сици́лия", "Сицилия")},
    "Toscana"   : {"it": ("Toscana", "Toscana"), "en": ("Tuscany", "Tuscany"), "ru": ("Тоска́на", "Тоскана")},
    "Trentino-Alto Adige"   : {"it": ("Trentino-Alto Adige", "Trentino-Alto Adige"), "en": ("Trentino-Alto Adige/Südtirol", "Trentino-Alto Adige/Südtirol"), "ru": ("Тренти́но-А́льто-А́дидже", "Трентино-Альто-Адидже")},
    "Umbria"   : {"it": ("Umbria", "Umbria"), "en": ("Umbria", "Umbria"), "ru": ("У́мбрия", "Умбрия")},
    "Valle d'Aosta"   : {"it": ("Valle d'Aosta", "Valle d'Aosta"), "en": ("Aosta Valley", "Aosta Valley"), "ru": ("Ва́лле-д’Ао́ста", "Валле-д’Аоста")},
    "Veneto"   : {"it": ("Veneto", "Veneto"), "en": ("Veneto", "Veneto"), "ru": ("Вене́ция (Ве́нето)", "Венеция (область)")}
}

In [18]:
# create code for placing info in Wikipedia
def create_table_regions_v1(df, file_header, lang='ru'):

    def if_value(x, prec=1):
        return '—' if math.isnan(x) else \
               f"{x:0.{prec}f}"  if x>=0 else \
               f"−{-x:0.{prec}f}"                #"{x:0.{prec}f}".format(x, prec)
    
    def chval(x, prec=1, *, add_par=''):  # change_value
        return f'style="background:#fffae0;padding-right:1.5ex;{add_par}"|—' if math.isnan(x) else \
               f'style="background:#fffae0;padding-right:1.5ex;color:darkgreen;{add_par}"|{x:0.{prec}f}' if x>0 else \
               f'style="background:#fffae0;padding-right:1.5ex;color:crimson;{add_par}"|−{-x:0.{prec}f}' if x<0 else \
               f'style="background:#fffae0;padding-right:1.5ex;color:darkgray;{add_par}"|{x:0.{prec}f}'
    
    def chval_bold(x, prec=1, *, add_par=''):  # change_value
        return f'style="background:#fffae0;padding-right:1.5ex;{add_par}"|\'\'\'—\'\'\'' if math.isnan(x) else \
               f'style="background:#fffae0;padding-right:1.5ex;color:darkgreen;{add_par}"|\'\'\'{x:0.{prec}f}\'\'\'' if x>0 else \
               f'style="background:#fffae0;padding-right:1.5ex;color:crimson;{add_par}"|\'\'\'−{-x:0.{prec}f}\'\'\'' if x<0 else \
               f'style="background:#fffae0;padding-right:1.5ex;color:darkgray;{add_par}"|\'\'\'{x:0.{prec}f}\'\'\''

    with open('design/' + file_header, mode='r', encoding="utf-8") as fh:
        table_header = fh.read()

    st = ''
    for i in range(len(df)):
        ser = df.iloc[i]
        if ser.name == 'ITALIA':
             st += '\n' + '|-class=static-row-header\n' + \
                  f'| \'\'\'{dd_replacement_regions[ser.name][lang][0]}\'\'\' ' + \
                  f'||style="background:#e0ffd8;"| \'\'\'{if_value(ser["2024"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{if_value(ser["m_2024"])}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{if_value(ser["f_2024"])}\'\'\' ' + \
                  f'||style="background:#fff8dc;"| \'\'\'{if_value(ser["f-m_2024"])}\'\'\' ' + \
                  f'||style="border-left-width:2px;"| \'\'\'{if_value(ser["2014"])}\'\'\' ' + \
                  f'||{chval_bold(ser["2014→2019"])} ' + \
                  f'|| \'\'\'{if_value(ser["2019"])}\'\'\' ' + \
                  f'||{chval_bold(ser["2019→2020"])} ' + \
                  f'|| \'\'\'{if_value(ser["2020"])}\'\'\' ' + \
                  f'||{chval_bold(ser["2020→2021"])} ' + \
                  f'|| \'\'\'{if_value(ser["2021"])}\'\'\' ' + \
                  f'||{chval_bold(ser["2021→2022"])} ' + \
                  f'|| \'\'\'{if_value(ser["2022"])}\'\'\' ' + \
                  f'||{chval_bold(ser["2022→2023"])} ' + \
                  f'|| \'\'\'{if_value(ser["2023"])}\'\'\' ' + \
                  f'||{chval_bold(ser["2023→2024"])} ' + \
                  f'||style="background:#e0ffd8;"| \'\'\'{if_value(ser["2024"])}\'\'\' ' + \
                  f'||{chval_bold(ser["2014→2024"], add_par="border-left-width:2px;")}'
        else:
            name_link = dd_replacement_regions[ser.name][lang][1]
            name_visible = dd_replacement_regions[ser.name][lang][0]
            name_inserted = name_link if name_link == name_visible else f"{name_link}|{name_visible}"
            st += '\n' + '|-\n' + \
                  f'| [[{name_inserted}]] ' + \
                  f'||style="background:#e0ffd8;"| \'\'\'{if_value(ser["2024"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| {if_value(ser["m_2024"])} ' + \
                  f'||style="background:#fee7f6;"| {if_value(ser["f_2024"])} ' + \
                  f'||style="background:#fff8dc;"| {if_value(ser["f-m_2024"])} ' + \
                  f'||style="border-left-width:2px;"| {if_value(ser["2014"])} ' + \
                  f'||{chval(ser["2014→2019"])} ' + \
                  f'|| {if_value(ser["2019"])} ' + \
                  f'||{chval(ser["2019→2020"])} ' + \
                  f'|| {if_value(ser["2020"])} ' + \
                  f'||{chval(ser["2020→2021"])} ' + \
                  f'|| {if_value(ser["2021"])} ' + \
                  f'||{chval(ser["2021→2022"])} ' + \
                  f'|| {if_value(ser["2022"])} ' + \
                  f'||{chval(ser["2022→2023"])} ' + \
                  f'|| {if_value(ser["2023"])} ' + \
                  f'||{chval(ser["2023→2024"])} ' + \
                  f'||style="background:#e0ffd8;"| \'\'\'{if_value(ser["2024"])}\'\'\' ' + \
                  f'||{chval(ser["2014→2024"], add_par="border-left-width:2px;")}'

    if lang == 'ru':
        st = re.sub('(?<=\\d)\\.(?=\\d)', ',', st)  # replace . to comma, if this . is between two digits
        st = st.replace('padding-right:1,5ex;', 'padding-right:1.5ex;')

    st = table_header + st + '\n|}'
    
    # gray color for missing values
    st = st.replace(';"| —', ';color:silver;"| —')

    return st


table_code = create_table_regions_v1(df_regions, file_header='Italian_regions_header_ru -2024 -v1.txt', lang='ru')
# write the code to file
with open('output/Table code for Italian regions -ru -v1.txt', 'w', encoding="utf-8") as fh:
    fh.write(table_code)


table_code = create_table_regions_v1(df_regions, file_header='Italian_regions_header_en -2024 -v1.txt', lang='en')
# write the code to file
with open('output/Table code for Italian regions -en -v1.txt', 'w', encoding="utf-8") as fh:
    fh.write(table_code)

In [19]:
# alternative version of the table
def create_table_regions_v2(df, file_header, lang='ru'):

    def if_value(x, prec=1):
        return '—' if math.isnan(x) else \
               f"{x:0.{prec}f}"  if x>=0 else \
               f"−{-x:0.{prec}f}"                #"{x:0.{prec}f}".format(x, prec)
    
    def chval(x, prec=1, *, add_par=''):  # change_value
        return f'style="padding-right:1.5ex;{add_par}"|—' if math.isnan(x) else \
               f'style="padding-right:1.5ex;color:darkgreen;{add_par}"|{x:0.{prec}f}' if x>0 else \
               f'style="padding-right:1.5ex;color:crimson;{add_par}"|−{-x:0.{prec}f}' if x<0 else \
               f'style="padding-right:1.5ex;color:darkgray;{add_par}"|{x:0.{prec}f}'
    
    def chval_bold(x, prec=1, *, add_par=''):  # change_value
        return f'style="padding-right:1.5ex;{add_par}"|\'\'\'—\'\'\'' if math.isnan(x) else \
               f'style="padding-right:1.5ex;color:darkgreen;{add_par}"|\'\'\'{x:0.{prec}f}\'\'\'' if x>0 else \
               f'style="padding-right:1.5ex;color:crimson;{add_par}"|\'\'\'−{-x:0.{prec}f}\'\'\'' if x<0 else \
               f'style="padding-right:1.5ex;color:darkgray;{add_par}"|\'\'\'{x:0.{prec}f}\'\'\''

    with open('design/' + file_header, mode='r', encoding="utf-8") as fh:
        table_header = fh.read()

    st = ''
    for i in range(len(df)):
        ser = df.iloc[i]
        if ser.name == 'ITALIA':
             st += '\n' + '|-class=static-row-header\n' + \
                  f'| \'\'\'{dd_replacement_regions[ser.name][lang][0]}\'\'\' ' + \
                  f'||style="background:#e0ffd8;"| \'\'\'{if_value(ser["2014"])} ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{if_value(ser["m_2014"])} ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{if_value(ser["f_2014"])} ' + \
                  f'|| \'\'\'{if_value(ser["f-m_2014"])}\'\'\' ' + \
                  f'||{chval_bold(ser["2014→2019"], add_par="border-left-width:2px;")} ' + \
                  f'||style="border-left-width:2px;background:#e0ffd8;"| \'\'\'{if_value(ser["2019"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{if_value(ser["m_2019"])}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{if_value(ser["f_2019"])}\'\'\' ' + \
                  f'|| \'\'\'{if_value(ser["f-m_2019"])}\'\'\' ' + \
                  f'||{chval_bold(ser["2019→2024"], add_par="border-left-width:2px;")} ' + \
                  f'||style="border-left-width:2px;background:#e0ffd8;"| \'\'\'{if_value(ser["2024"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{if_value(ser["m_2024"])}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{if_value(ser["f_2024"])}\'\'\' ' + \
                  f'|| \'\'\'{if_value(ser["f-m_2024"])}\'\'\' ' + \
                  f'||{chval_bold(ser["2014→2024"], add_par="border-left-width:2px;")}'
        else:
            name_link = dd_replacement_regions[ser.name][lang][1]
            name_visible = dd_replacement_regions[ser.name][lang][0]
            name_inserted = name_link if name_link == name_visible else f"{name_link}|{name_visible}"
            st += '\n' + '|-\n' + \
                  f'| [[{name_inserted}]] ' + \
                  f'||style="background:#e0ffd8;"| {if_value(ser["2014"])} ' + \
                  f'||style="background:#eaf3ff;"| {if_value(ser["m_2014"])} ' + \
                  f'||style="background:#fee7f6;"| {if_value(ser["f_2014"])} ' + \
                  f'|| {if_value(ser["f-m_2014"])} ' + \
                  f'||{chval(ser["2014→2019"], add_par="border-left-width:2px;")} ' + \
                  f'||style="border-left-width:2px;background:#e0ffd8;"| {if_value(ser["2019"])} ' + \
                  f'||style="background:#eaf3ff;"| {if_value(ser["m_2019"])} ' + \
                  f'||style="background:#fee7f6;"| {if_value(ser["f_2019"])} ' + \
                  f'|| {if_value(ser["f-m_2019"])} ' + \
                  f'||{chval(ser["2019→2024"], add_par="border-left-width:2px;")} ' + \
                  f'||style="border-left-width:2px;background:#e0ffd8;"| {if_value(ser["2024"])} ' + \
                  f'||style="background:#eaf3ff;"| {if_value(ser["m_2024"])} ' + \
                  f'||style="background:#fee7f6;"| {if_value(ser["f_2024"])} ' + \
                  f'|| {if_value(ser["f-m_2024"])} ' + \
                  f'||{chval(ser["2014→2024"], add_par="border-left-width:2px;")}'

    if lang == 'ru':
        st = re.sub('(?<=\\d)\\.(?=\\d)', ',', st)  # replace . to comma, if this . is between two digits
        st = st.replace('padding-right:1,5ex;', 'padding-right:1.5ex;')

    st = table_header + st + '\n|}'
    
    # gray color for missing values
    st = st.replace(';"| —', ';color:silver;"| —')

    return st


table_code = create_table_regions_v2(df_regions, file_header='Italian_regions_header_ru -2024 -v2.txt', lang='ru')
# write the code to file
with open('output/Table code for Italian regions -ru -v2.txt', 'w', encoding="utf-8") as fh:
    fh.write(table_code)


table_code = create_table_regions_v2(df_regions, file_header='Italian_regions_header_en -2024 -v2.txt', lang='en')
# write the code to file
with open('output/Table code for Italian regions -en -v2.txt', 'w', encoding="utf-8") as fh:
    fh.write(table_code)

<br />
<br />
<br />

---
<h4>Statistics for provinces</h4>

In [21]:
df_provinces = df.drop(regions + macro_regions) \
                 .loc[:, cols_to_table]

# df_provinces.columns = ['all', 'male', 'female', '2019', '2020', '2021', '2022']

df_provinces

,2014,2019,2020,2021,2022,2023,2024,m_2014,m_2019,m_2024,f_2014,f_2019,f_2024
,,,,,,,,,,,,,
Torino,83.0,83.3,81.8,82.8,82.8,83.4,83.9,80.9,81.2,81.9,85.2,85.5,85.9
Vercelli,81.7,82.4,80.2,81.6,81.7,82.0,82.6,79.0,80.6,80.4,84.5,84.2,84.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...
Sud Sardegna,81.5,82.8,82.4,82.2,82.1,82.1,82.7,79.2,79.9,80.2,84.0,85.8,85.2
ITALIA,82.6,83.2,82.1,82.5,82.6,83.0,83.5,80.3,81.1,81.5,85.0,85.4,85.6


In [22]:
extend_df(df_provinces)

df_provinces = sort_df(df_provinces)

df_provinces

,2014,2014→2019,2019,2019→2020,2020,2020→2021,2021,2021→2022,2022,2022→2023,2023,2023→2024,2024,2019→2024,2014→2024,m_2014,m_2019,m_2024,f_2014,f_2019,f_2024,f-m_2014,f-m_2019,f-m_2024
,,,,,,,,,,,,,,,,,,,,,,,,
ITALIA,82.6,0.6,83.2,-1.1,82.1,0.4,82.5,0.1,82.6,0.4,83.0,0.5,83.5,0.3,0.9,80.3,81.1,81.5,85.0,85.4,85.6,4.7,4.3,4.1
Treviso,84.0,0.3,84.3,-0.7,83.6,0.3,83.9,0.2,84.1,0.3,84.4,0.6,85.0,0.7,1.0,81.6,82.2,83.0,86.4,86.5,87.0,4.8,4.3,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Napoli,80.3,1.0,81.3,-0.9,80.4,-0.1,80.3,0.4,80.7,0.4,81.1,0.5,81.6,0.3,1.3,78.0,79.3,79.6,82.8,83.4,83.6,4.8,4.1,4.0
Caserta,80.8,0.7,81.5,-0.8,80.7,0.0,80.7,0.1,80.8,0.1,80.9,0.6,81.5,0.0,0.7,78.6,79.3,79.3,83.2,83.9,83.8,4.6,4.6,4.5


<br />
<br />

In [24]:
mal.min_and_max_values(
    df_provinces[['2014', '2014→2019', '2019', '2019→2024', '2024', '2014→2024']],
    row_center='ITALIA',  max_lng=13)

Number of records: 108


,2014,2014→2019,2019,2019→2024,2024,2014→2024
max,84.0 -Treviso,1.4 -Enna,84.5 -Prato,1.0 -Gorizia,85.0 -Treviso,1.9 -Enna
max_2,84.0 -Firenze,1.3 -Lodi,84.4 -Pordenone,0.9 -Caltanissetta,84.9 -Lecco,1.8 -Gorizia
max_3,83.8 -Milano,1.3 -Rovigo,84.4 -Perugia,0.8 -Bolzano,84.8 -Monza e dell…,1.5 -Lecco
ITALIA,– 82.6 –,– 0.6 –,– 83.2 –,– 0.3 –,– 83.5 –,– 0.9 –
min_3,80.9 -Enna,0.0 -Trapani,81.5 -Siracusa,-0.4 -Oristano,81.6 -Napoli,0.2 -Agrigento
min_2,80.8 -Caserta,-0.1 -Potenza,81.3 -Napoli,-0.5 -Rieti,81.6 -Siracusa,0.2 -Rieti
min,80.3 -Napoli,-0.2 -Agrigento,81.1 -Caltanissetta,-0.6 -Campobasso,81.5 -Caserta,0.1 -Grosseto


In [25]:
mal.min_and_max_values(
    df_provinces[['m_2014', 'm_2019', 'm_2024', 'f_2014', 'f_2019', 'f_2024', 'f-m_2014', 'f-m_2019', 'f-m_2024']],
    row_center='ITALIA',  max_lng=13)

Number of records: 108


,m_2014,m_2019,m_2024,f_2014,f_2019,f_2024,f-m_2014,f-m_2019,f-m_2024
max,81.9 -Firenze,82.7 -Prato,83.4 -Lecco,86.4 -Treviso,86.7 -Trento,87.0 -Treviso,6.5 -Nuoro,6.6 -Nuoro,6.6 -Nuoro
max_2,81.8 -Prato,82.6 -Pordenone,83.1 -Monza e dell…,86.3 -Firenze,86.5 -Treviso,86.9 -Trento,6.2 -Sondrio,5.9 -Sud Sardegna,5.3 -Oristano
max_3,81.6 -Treviso,82.5 -Rimini,83.0 -Treviso,86.2 -Trento,86.5 -Firenze,86.8 -Verona,5.9 -Campobasso,5.7 -Aosta,5.1 -Sondrio
ITALIA,– 80.3 –,– 81.1 –,– 81.5 –,– 85.0 –,– 85.4 –,– 85.6 –,– 4.7 –,– 4.3 –,– 4.1 –
min_3,78.7 -Nuoro,79.3 -Caserta,79.7 -Nuoro,83.0 -Caltanissetta,83.7 -Siracusa,83.8 -Caltanissetta,3.9 -Bari,3.5 -Grosseto,3.2 -Bologna
min_2,78.6 -Caserta,79.3 -Napoli,79.6 -Napoli,82.8 -Napoli,83.4 -Napoli,83.6 -Napoli,3.9 -Prato,3.4 -Barletta-And…,3.0 -Lecco
min,78.0 -Napoli,79.1 -Caltanissetta,79.3 -Caserta,82.8 -Enna,83.3 -Caltanissetta,83.5 -Siracusa,3.8 -Enna,3.4 -Arezzo,2.9 -Ravenna


<br />
<br />

In [27]:
dd_replacement_provinces = {
    "ITALIA"   : {"it": ("Italia", ""), "en": ("Italy on average", ""), "ru": ("Италия в среднем", "")},
      # metropolitan cities
    "Bari"     : {"it": ("Bari", "Città metropolitana di Bari"), "en": ("Bari", "Metropolitan City of Bari"), "ru": ("Ба́ри", "Бари (метрополитенский город)")},
    "Bologna"  : {"it": ("Bologna", "Città metropolitana di Bologna"), "en": ("Bologna", "Metropolitan City of Bologna"), "ru": ("Боло́нья", "Болонья (метрополитенский город)")},
    "Cagliari" : {"it": ("Cagliari", "Città metropolitana di Cagliari"), "en": ("Cagliari", "Metropolitan City of Cagliari"), "ru": ("Ка́льяри", "Кальяри (метрополитенский город)")},
    "Catania"  : {"it": ("Catania", "Città metropolitana di Catania"), "en": ("Catania", "Metropolitan City of Catania"), "ru": ("Ката́ния", "Катания (метрополитенский город)")},
    "Firenze"  : {"it": ("Firenze", "Città metropolitana di Firenze"), "en": ("Florence", "Metropolitan City of Florence"), "ru": ("Флоре́нция", "Флоренция (метрополитенский город)")},
    "Genova"   : {"it": ("Genova", "Città metropolitana di Genova"), "en": ("Genoa", "Metropolitan City of Genoa"), "ru": ("Ге́нуя", "Генуя (метрополитенский город)")},
    "Messina"  : {"it": ("Messina", "Città metropolitana di Messina"), "en": ("Messina", "Metropolitan City of Messina"), "ru": ("Месси́на", "Мессина (метрополитенский город)")},
    "Milano"   : {"it": ("Milano", "Città metropolitana di Milano"), "en": ("Milan", "Metropolitan City of Milan"), "ru": ("Мила́н", "Милан (метрополитенский город)")},
    "Napoli"   : {"it": ("Napoli", "Città metropolitana di Napoli"), "en": ("Naples", "Metropolitan City of Naples"), "ru": ("Неа́поль", "Неаполь (метрополитенский город)")},
    "Palermo"  : {"it": ("Palermo", "Città metropolitana di Palermo"), "en": ("Palermo", "Metropolitan City of Palermo"), "ru": ("Пале́рмо", "Палермо (метрополитенский город)")},
    "Reggio di Calabria"   : {"it": ("Reggio Calabria", "Città metropolitana di Reggio Calabria"), "en": ("Reggio Calabria", "Metropolitan City of Reggio Calabria"), "ru": ("Ре́джо-ди-Кала́брия", "Реджо-ди-Калабрия (метрополитенский город)")},
    "Roma"     : {"it": ("Roma", "Città metropolitana di Roma Capitale"), "en": ("Rome", "Metropolitan City of Rome Capital"), "ru": ("Рим", "Рим (метрополитенский город)")},
    "Torino"   : {"it": ("Torino", "Città metropolitana di Torino"), "en": ("Turin", "Metropolitan City of Turin"), "ru": ("Тури́н", "Турин (метропольный город)")},
    "Venezia"  : {"it": ("Venezia", "Venezia"), "en": ("Venice", "Metropolitan City of Venice"), "ru": ("Вене́ция", "Венеция (метрополитенский город)")},
      # provinces
    "Agrigento": {"it": ("Agrigento", "Provincia di Agrigento"), "en": ("Agrigento", "Province of Agrigento"), "ru": ("Агридже́нто", "Агридженто (провинция)")},
    "Alessandria" : {"it": ("Alessandria", "Provincia di Alessandria"), "en": ("Alessandria", "Province of Alessandria"), "ru": ("Алесса́ндрия", "Алессандрия (провинция)")},
    "Ancona"   : {"it": ("Ancona", "Provincia di Ancona"), "en": ("Ancona", "Province of Ancona"), "ru": ("Анко́на", "Анкона (провинция)")},
    "Aosta"    : {"it": ("Valle d'Aosta", "Valle d'Aosta"), "en": ("Aosta", "Aosta Valley"), "ru": ("Ва́лле-д’Ао́ста", "Валле-д’Аоста")},
    "Arezzo"   : {"it": ("Arezzo", "Provincia di Arezzo"), "en": ("Arezzo", "Province of Arezzo"), "ru": ("Аре́ццо", "Ареццо (провинция)")},
    "Ascoli Piceno"   : {"it": ("Ascoli Piceno", "Provincia di Ascoli Piceno"), "en": ("Ascoli Piceno", "Province of Ascoli Piceno"), "ru": ("А́сколи-Пиче́но", "Асколи-Пичено (провинция)")},
    "Asti"     : {"it": ("Asti", "Provincia di Asti"), "en": ("Asti", "Province of Asti"), "ru": ("А́сти", "Асти (провинция)")},
    "Avellino" : {"it": ("Avellino", "Provincia di Avellino"), "en": ("Avellino", "Province of Avellino"), "ru": ("Авелли́но", "Авеллино (провинция)")},
    "Barletta-Andria-Trani" : {"it": ("Barletta-Andria-Trani", "Provincia di Barletta-Andria-Trani"), "en": ("Barletta-Andria-Trani", "Province of Barletta-Andria-Trani"), "ru": ("Барле́тта-А́ндрия-Тра́ни", "Барлетта-Андрия-Трани")},
    "Belluno"  : {"it": ("Belluno", "Provincia di Belluno"), "en": ("Belluno", "Province of Belluno"), "ru": ("Беллу́но", "Беллуно (провинция)")},
    "Benevento": {"it": ("Benevento", "Provincia di Benevento"), "en": ("Benevento", "Province of Benevento"), "ru": ("Беневе́нто", "Беневенто (провинция)")},
    "Bergamo"  : {"it": ("Bergamo", "Provincia di Bergamo"), "en": ("Bergamo", "Province of Bergamo"), "ru": ("Бе́ргамо", "Бергамо (провинция)")},
    "Biella"   : {"it": ("Biella", "Provincia di Biella"), "en": ("Biella", "Province of Biella"), "ru": ("Биелла", "Биелла (провинция)")},
    "Bolzano"  : {"it": ("Bolzano", "Provincia autonoma di Bolzano"), "en": ("South Tyrol (Bolzano)", "South Tyrol"), "ru": ("Южный Тиро́ль (Больца́но)", "Южный Тироль")},
    "Brescia"  : {"it": ("Brescia", "Provincia di Brescia"), "en": ("Brescia", "Province of Brescia"), "ru": ("Бре́шиа", "Брешиа (провинция)")},
    "Brindisi" : {"it": ("Brindisi", "Provincia di Brindisi"), "en": ("Brindisi", "Province of Brindisi"), "ru": ("Бри́ндизи", "Бриндизи (провинция)")},
    "Caltanissetta"   : {"it": ("Caltanissetta", "Provincia di Caltanissetta"), "en": ("Caltanissetta", "Province of Caltanissetta"), "ru": ("Кальтаниссетта", "Кальтаниссетта (провинция)")},
    "Campobasso" : {"it": ("Campobasso", "Provincia di Campobasso"), "en": ("Campobasso", "Province of Campobasso"), "ru": ("Кампоба́ссо", "Кампобассо (провинция)")},
    "Caserta"  : {"it": ("Caserta", "Provincia di Caserta"), "en": ("Caserta", "Province of Caserta"), "ru": ("Казе́рта", "Казерта (провинция)")},
    "Catanzaro": {"it": ("Catanzaro", "Provincia di Catanzaro"), "en": ("Catanzaro", "Province of Catanzaro"), "ru": ("Катандза́ро", "Катандзаро (провинция)")},
    "Chieti"   : {"it": ("Chieti", "Provincia di Chieti"), "en": ("Chieti", "Province of Chieti"), "ru": ("Кье́ти", "Кьети (провинция)")},
    "Como"     : {"it": ("Como", "Provincia di Como"), "en": ("Como", "Province of Como"), "ru": ("Ко́мо", "Комо (провинция)")},
    "Cosenza"  : {"it": ("Cosenza", "Provincia di Cosenza"), "en": ("Cosenza", "Province of Cosenza"), "ru": ("Козе́нца", "Козенца (провинция)")},
    "Cremona"  : {"it": ("Cremona", "Provincia di Cremona"), "en": ("Cremona", "Province of Cremona"), "ru": ("Кремо́на", "Кремона (провинция)")},
    "Crotone"  : {"it": ("Crotone", "Provincia di Crotone"), "en": ("Crotone", "Province of Crotone"), "ru": ("Крото́не", "Кротоне (провинция)")},
    "Cuneo"    : {"it": ("Cuneo", "Provincia di Cuneo"), "en": ("Cuneo", "Province of Cuneo"), "ru": ("Ку́нео", "Кунео (провинция)")},
    "Enna"     : {"it": ("Enna", "Provincia di Enna"), "en": ("Enna", "Province of Enna"), "ru": ("Э́нна", "Энна (провинция)")},
    "Fermo"    : {"it": ("Fermo", "Provincia di Fermo"), "en": ("Fermo", "Province of Fermo"), "ru": ("Фе́рмо", "Фермо (провинция)")},
    "Ferrara"  : {"it": ("Ferrara", "Provincia di Ferrara"), "en": ("Ferrara", "Province of Ferrara"), "ru": ("Ферра́ра", "Феррара (провинция)")},
    "Foggia"   : {"it": ("Foggia", "Provincia di Foggia"), "en": ("Foggia", "Province of Foggia"), "ru": ("Фо́джа", "Фоджа (провинция)")},
    "Forli'"   : {"it": ("Forlì-Cesena", "Provincia di Forlì-Cesena"), "en": ("Forlì-Cesena", "Province of Forlì-Cesena"), "ru": ("Форли́-Чезе́на", "Форли-Чезена")},
    "Frosinone": {"it": ("Frosinone", "Provincia di Frosinone"), "en": ("Frosinone", "Province of Frosinone"), "ru": ("Фрозиноне", "Фрозиноне (провинция)")},
    "Gorizia"  : {"it": ("Gorizia", "Provincia di Gorizia"), "en": ("Gorizia", "Province of Gorizia"), "ru": ("Гори́ция", "Гориция (провинция)")},
    "Grosseto" : {"it": ("Grosseto", "Provincia di Grosseto"), "en": ("Grosseto", "Province of Grosseto"), "ru": ("Гроссе́то", "Гроссето (провинция)")},
    "Imperia"  : {"it": ("Imperia", "Provincia di Imperia"), "en": ("Imperia", "Province of Imperia"), "ru": ("Импе́рия", "Империя (провинция)")},
    "Isernia"  : {"it": ("Isernia", "Provincia di Isernia"), "en": ("Isernia", "Province of Isernia"), "ru": ("Изе́рния", "Изерния (провинция)")},
    "L'Aquila" : {"it": ("L'Aquila", "Provincia dell'Aquila"), "en": ("L'Aquila", "Province of L'Aquila"), "ru": ("Л’А́куила", "Л’Акуила (провинция)")},
    "La Spezia": {"it": ("La Spezia", "Provincia della Spezia"), "en": ("La Spezia", "Province of La Spezia"), "ru": ("Спе́ция", "Специя (провинция)")},
    "Latina"   : {"it": ("Latina", "Provincia di Latina"), "en": ("Latina", "Province of Latina"), "ru": ("Лати́на", "Латина (провинция)")},
    "Lecce"    : {"it": ("Lecce", "Provincia di Lecce"), "en": ("Lecce", "Province of Lecce"), "ru": ("Ле́чче", "Лечче (провинция)")},
    "Lecco"    : {"it": ("Lecco", "Provincia di Lecco"), "en": ("Lecco", "Province of Lecco"), "ru": ("Ле́кко", "Лекко (провинция)")},
    "Livorno"  : {"it": ("Livorno", "Provincia di Livorno"), "en": ("Livorno", "Province of Livorno"), "ru": ("Ливо́рно", "Ливорно (провинция)")},
    "Lodi"     : {"it": ("Lodi", "Provincia di Lodi"), "en": ("Lodi", "Province of Lodi"), "ru": ("Ло́ди", "Лоди (провинция)")},
    "Lucca"    : {"it": ("Lucca", "Provincia di Lucca"), "en": ("Lucca", "Province of Lucca"), "ru": ("Лу́кка", "Лукка (провинция)")},
    "Macerata" : {"it": ("Macerata", "Provincia di Macerata"), "en": ("Macerata", "Province of Macerata"), "ru": ("Мачера́та", "Мачерата (провинция)")},
    "Mantova"  : {"it": ("Mantova", "Provincia di Mantova"), "en": ("Mantua", "Province of Mantua"), "ru": ("Ма́нтуя", "Мантуя (провинция)")},
    "Massa-Carrara" : {"it": ("Massa-Carrara", "Provincia di Massa-Carrara"), "en": ("Massa-Carrara", "Province of Massa-Carrara"), "ru": ("Ма́сса-Карра́ра", "Масса-Каррара (провинция)")},
    "Matera"   : {"it": ("Matera", "Provincia di Matera"), "en": ("Matera", "Province of Matera"), "ru": ("Мате́ра", "Матера (провинция)")},
    "Modena"   : {"it": ("Modena", "Provincia di Modena"), "en": ("Modena", "Province of Modena"), "ru": ("Мо́дена", "Модена (провинция)")},
    "Monza e della Brianza"   : {"it": ("Monza e della Brianza", "Provincia di Monza e della Brianza"), "en": ("Monza and Brianza", "Province of Monza and Brianza"), "ru": ("Мо́нца-э-Бриа́нца", "Монца-э-Брианца")},
    "Novara"   : {"it": ("Novara", "Provincia di Novara"), "en": ("Novara", "Province of Novara"), "ru": ("Нова́ра", "Новара (провинция)")},
    "Nuoro"    : {"it": ("Nuoro", "Provincia di Nuoro"), "en": ("Nuoro", "Province of Nuoro"), "ru": ("Ну́оро", "Нуоро (провинция)")},
    "Oristano" : {"it": ("Oristano", "Provincia di Oristano"), "en": ("Oristano", "Province of Oristano"), "ru": ("Ориста́но", "Ористано (провинция)")},
    "Padova"   : {"it": ("Padova", "Provincia di Padova"), "en": ("Padua", "Province of Padua"), "ru": ("Па́дуя (Па́дова)", "Падуя (провинция)")},
    "Parma"    : {"it": ("Parma", "Provincia di Parma"), "en": ("Parma", "Province of Parma"), "ru": ("Па́рма", "Парма (провинция)")},
    "Pavia"    : {"it": ("Pavia", "Provincia di Pavia"), "en": ("Pavia", "Province of Pavia"), "ru": ("Пави́я", "Павия (провинция)")},
    "Perugia"  : {"it": ("Perugia", "Provincia di Perugia"), "en": ("Perugia", "Province of Perugia"), "ru": ("Перу́джа", "Перуджа (провинция)")},
    "Pesaro e Urbino" : {"it": ("Pesaro e Urbino", "Provincia di Pesaro e Urbino"), "en": ("Pesaro and Urbino", "Province of Pesaro and Urbino"), "ru": ("Пе́заро-э-Урби́но", "Пезаро-э-Урбино")},
    "Pescara"  : {"it": ("Pescara", "Provincia di Pescara"), "en": ("Pescara", "Province of Pescara"), "ru": ("Песка́ра", "Пескара (провинция)")},
    "Piacenza" : {"it": ("Piacenza", "Provincia di Piacenza"), "en": ("Piacenza", "Province of Piacenza"), "ru": ("Пьяче́нца", "Пьяченца (провинция)")},
    "Pisa"     : {"it": ("Pisa", "Provincia di Pisa"), "en": ("Pisa", "Province of Pisa"), "ru": ("Пи́за", "Пиза (провинция)")},
    "Pistoia"  : {"it": ("Pistoia", "Provincia di Pistoia"), "en": ("Pistoia", "Province of Pistoia"), "ru": ("Писто́я", "Пистоя (провинция)")},
    "Pordenone": {"it": ("Pordenone", "Provincia di Pordenone"), "en": ("Pordenone", "Province of Pordenone"), "ru": ("Пордено́не", "Порденоне (провинция)")},
    "Potenza"  : {"it": ("Potenza", "Provincia di Potenza"), "en": ("Potenza", "Province of Potenza"), "ru": ("Поте́нца", "Потенца (провинция)")},
    "Prato"    : {"it": ("Prato", "Provincia di Prato"), "en": ("Prato", "Province of Prato"), "ru": ("Пра́то", "Прато (провинция)")},
    "Ragusa"   : {"it": ("Ragusa", "Provincia di Ragusa"), "en": ("Ragusa", "Province of Ragusa"), "ru": ("Рагу́за", "Рагуза (провинция)")},
    "Ravenna"  : {"it": ("Ravenna", "Provincia di Ravenna"), "en": ("Ravenna", "Province of Ravenna"), "ru": ("Раве́нна", "Равенна (провинция)")},
    "Reggio nell'Emilia"   : {"it": ("Reggio Emilia", "Provincia di Reggio Emilia"), "en": ("Reggio Emilia", "Province of Reggio Emilia"), "ru": ("Ре́джо-нель-Эми́лия", "Реджо-нель-Эмилия (провинция)")},
    "Rieti"    : {"it": ("Rieti", "Provincia di Rieti"), "en": ("Rieti", "Province of Rieti"), "ru": ("Риети", "Риети (провинция)")},
    "Rimini"   : {"it": ("Rimini", "Provincia di Rimini"), "en": ("Rimini", "Province of Rimini"), "ru": ("Ри́мини", "Римини (провинция)")},
    "Rovigo"   : {"it": ("Rovigo", "Provincia di Rovigo"), "en": ("Rovigo", "Province of Rovigo"), "ru": ("Рови́го", "Ровиго (провинция)")},
    "Salerno"  : {"it": ("Salerno", "Provincia di Salerno"), "en": ("Salerno", "Province of Salerno"), "ru": ("Сале́рно", "Салерно (провинция)")},
    "Sassari"  : {"it": ("Sassari", "Provincia di Sassari"), "en": ("Sassari", "Provincia di Salerno"), "ru": ("Са́ссари", "Сассари (провинция)")},
    "Savona"   : {"it": ("Savona", "Provincia di Savona"), "en": ("Savona", "Province of Savona"), "ru": ("Саво́на", "Савона (провинция)")},
    "Siena"    : {"it": ("Siena", "Provincia di Siena"), "en": ("Siena", "Province of Siena"), "ru": ("Сие́на", "Сиена (провинция)")},
    "Siracusa" : {"it": ("Siracusa", "Provincia di Siracusa"), "en": ("Syracuse", "Province of Syracuse"), "ru": ("Сираку́за", "Сиракуза (провинция)")},
    "Sondrio"  : {"it": ("Sondrio", "Provincia di Sondrio"), "en": ("Sondrio", "Province of Sondrio"), "ru": ("Со́ндрио", "Сондрио (провинция)")},
    "Sud Sardegna" : {"it": ("Sud Sardegna", "Provincia del Sud Sardegna"), "en": ("South Sardinia", "Province of South Sardinia"), "ru": ("Южная Сарди́ния", "Южная Сардиния")},
    "Taranto"  : {"it": ("Taranto", "Provincia di Taranto"), "en": ("Taranto", "Province of Taranto"), "ru": ("Та́ранто", "Таранто (провинция)")},
    "Teramo"   : {"it": ("Teramo", "Provincia di Teramo"), "en": ("Teramo", "Province of Teramo"), "ru": ("Те́рамо", "Терамо (провинция)")},
    "Terni"    : {"it": ("Terni", "Provincia di Terni"), "en": ("Terni", "Province of Terni"), "ru": ("Те́рни", "Терни (провинция)")},
    "Trapani"  : {"it": ("Trapani", "Provincia di Trapani"), "en": ("Trapani", "Province of Trapani"), "ru": ("Тра́пани", "Трапани (провинция)")},
    "Trento"   : {"it": ("Trento (Trentino)", "Provincia autonoma di Trento"), "en": ("Trento (Trentino)", "Trentino"), "ru": ("Тре́нто (Трентино)", "Тренто (провинция)")},
    "Treviso"  : {"it": ("Treviso", "Provincia di Treviso"), "en": ("Treviso", "Province of Treviso"), "ru": ("Треви́зо", "Тревизо (провинция)")},
    "Trieste"  : {"it": ("Trieste", "Provincia di Trieste"), "en": ("Trieste", "Province of Trieste"), "ru": ("Трие́ст", "Триест (провинция)")},
    "Udine"    : {"it": ("Udine", "Provincia di Udine"), "en": ("Udine", "Province of Udine"), "ru": ("У́дине", "Удине (провинция)")},
    "Varese"   : {"it": ("Varese", "Provincia di Varese"), "en": ("Varese", "Province of Varese"), "ru": ("Варе́зе (Варе́се)", "Варесе (провинция)")},
    "Verbano-Cusio-Ossola" : {"it": ("Verbano-Cusio-Ossola", "Provincia del Verbano-Cusio-Ossola"), "en": ("Verbano-Cusio-Ossola", "Province of Verbano-Cusio-Ossola"), "ru": ("Вербано-Кузио-Оссола", "Вербано-Кузио-Оссола")},
    "Vercelli" : {"it": ("Vercelli", "Provincia di Vercelli"), "en": ("Vercelli", "Province of Vercelli"), "ru": ("Верче́лли", "Верчелли (провинция)")},
    "Verona"   : {"it": ("Verona", "Provincia di Verona"), "en": ("Verona", "Province of Verona"), "ru": ("Веро́на", "Верона (провинция)")},
    "Vibo Valentia" : {"it": ("Vibo Valentia", "Provincia di Vibo Valentia"), "en": ("Vibo Valentia", "Province of Vibo Valentia"), "ru": ("Вибо-Вале́нтия", "Вибо-Валентия (провинция)")},
    "Vicenza"  : {"it": ("Vicenza", "Provincia di Vicenza"), "en": ("Vicenza", "Province of Vicenza"), "ru": ("Виче́нца", "Виченца (провинция)")},
    "Viterbo"  : {"it": ("Viterbo", "Provincia di Viterbo"), "en": ("Viterbo", "Province of Viterbo"), "ru": ("Вите́рбо", "Витербо (провинция)")},
}

In [28]:
# create code for placing info in Wikipedia
def create_table_provinces_v1(df, file_header, lang='ru'):

    def if_value(x, prec=1):
        return '—' if math.isnan(x) else \
               f"{x:0.{prec}f}"  if x>=0 else \
               f"−{-x:0.{prec}f}"                #"{x:0.{prec}f}".format(x, prec)
    
    def chval(x, prec=1, *, add_par=''):  # change_value
        return f'style="background:#fffae0;padding-right:1.5ex;{add_par}"|—' if math.isnan(x) else \
               f'style="background:#fffae0;padding-right:1.5ex;color:darkgreen;{add_par}"|{x:0.{prec}f}' if x>0 else \
               f'style="background:#fffae0;padding-right:1.5ex;color:crimson;{add_par}"|−{-x:0.{prec}f}' if x<0 else \
               f'style="background:#fffae0;padding-right:1.5ex;color:darkgray;{add_par}"|{x:0.{prec}f}'
    
    def chval_bold(x, prec=1, *, add_par=''):  # change_value
        return f'style="background:#fffae0;padding-right:1.5ex;{add_par}"|\'\'\'—\'\'\'' if math.isnan(x) else \
               f'style="background:#fffae0;padding-right:1.5ex;color:darkgreen;{add_par}"|\'\'\'{x:0.{prec}f}\'\'\'' if x>0 else \
               f'style="background:#fffae0;padding-right:1.5ex;color:crimson;{add_par}"|\'\'\'−{-x:0.{prec}f}\'\'\'' if x<0 else \
               f'style="background:#fffae0;padding-right:1.5ex;color:darkgray;{add_par}"|\'\'\'{x:0.{prec}f}\'\'\''

    with open('design/' + file_header, mode='r', encoding="utf-8") as fh:
        table_header = fh.read()

    st = ''
    for i in range(len(df)):
        ser = df.iloc[i]
        if ser.name == 'ITALIA':
             st += '\n' + '|-class=static-row-header\n' + \
                  f'| \'\'\'{dd_replacement_provinces[ser.name][lang][0]}\'\'\' ' + \
                  f'||style="background:#e0ffd8;"| \'\'\'{if_value(ser["2024"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{if_value(ser["m_2024"])}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{if_value(ser["f_2024"])}\'\'\' ' + \
                  f'||style="background:#fff8dc;"| \'\'\'{if_value(ser["f-m_2024"])}\'\'\' ' + \
                  f'||style="border-left-width:2px;"| \'\'\'{if_value(ser["2014"])}\'\'\' ' + \
                  f'||{chval_bold(ser["2014→2019"])} ' + \
                  f'|| \'\'\'{if_value(ser["2019"])}\'\'\' ' + \
                  f'||{chval_bold(ser["2019→2020"])} ' + \
                  f'|| \'\'\'{if_value(ser["2020"])}\'\'\' ' + \
                  f'||{chval_bold(ser["2020→2021"])} ' + \
                  f'|| \'\'\'{if_value(ser["2021"])}\'\'\' ' + \
                  f'||{chval_bold(ser["2021→2022"])} ' + \
                  f'|| \'\'\'{if_value(ser["2022"])}\'\'\' ' + \
                  f'||{chval_bold(ser["2022→2023"])} ' + \
                  f'|| \'\'\'{if_value(ser["2023"])}\'\'\' ' + \
                  f'||{chval_bold(ser["2023→2024"])} ' + \
                  f'||style="background:#e0ffd8;"| \'\'\'{if_value(ser["2024"])}\'\'\' ' + \
                  f'||{chval_bold(ser["2014→2024"], add_par="border-left-width:2px;")}'
        else:
            name_link = dd_replacement_provinces[ser.name][lang][1]
            name_visible = dd_replacement_provinces[ser.name][lang][0]
            name_inserted = name_link if name_link == name_visible else f"{name_link}|{name_visible}"
            st += '\n' + '|-\n' + \
                  f'| [[{name_inserted}]] ' + \
                  f'||style="background:#e0ffd8;"| \'\'\'{if_value(ser["2024"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| {if_value(ser["m_2024"])} ' + \
                  f'||style="background:#fee7f6;"| {if_value(ser["f_2024"])} ' + \
                  f'||style="background:#fff8dc;"| {if_value(ser["f-m_2024"])} ' + \
                  f'||style="border-left-width:2px;"| {if_value(ser["2014"])} ' + \
                  f'||{chval(ser["2014→2019"])} ' + \
                  f'|| {if_value(ser["2019"])} ' + \
                  f'||{chval(ser["2019→2020"])} ' + \
                  f'|| {if_value(ser["2020"])} ' + \
                  f'||{chval(ser["2020→2021"])} ' + \
                  f'|| {if_value(ser["2021"])} ' + \
                  f'||{chval(ser["2021→2022"])} ' + \
                  f'|| {if_value(ser["2022"])} ' + \
                  f'||{chval(ser["2022→2023"])} ' + \
                  f'|| {if_value(ser["2023"])} ' + \
                  f'||{chval(ser["2023→2024"])} ' + \
                  f'||style="background:#e0ffd8;"| \'\'\'{if_value(ser["2024"])}\'\'\' ' + \
                  f'||{chval(ser["2014→2024"], add_par="border-left-width:2px;")}'

    if lang == 'ru':
        st = re.sub('(?<=\\d)\\.(?=\\d)', ',', st)  # replace . to comma, if this . is between two digits
        st = st.replace('padding-right:1,5ex;', 'padding-right:1.5ex;')

    st = table_header + st + '\n|}'
    
    # gray color for missing values
    st = st.replace(';"| —', ';color:silver;"| —')

    return st


table_code = create_table_provinces_v1(df_provinces, file_header='Italian_provinces_header_ru -2024 -v1.txt', lang='ru')
# write the code to file
with open('output/Table code for Italian provinces -ru -v1.txt', 'w', encoding="utf-8") as fh:
    fh.write(table_code)


table_code = create_table_provinces_v1(df_provinces, file_header='Italian_provinces_header_en -2024 -v1.txt', lang='en')
# write the code to file
with open('output/Table code for Italian provinces -en -v1.txt', 'w', encoding="utf-8") as fh:
    fh.write(table_code)

In [29]:
# alternative version of the table
def create_table_provinces_v2(df, file_header, lang='ru'):

    def if_value(x, prec=1):
        return '—' if math.isnan(x) else \
               f"{x:0.{prec}f}"  if x>=0 else \
               f"−{-x:0.{prec}f}"                #"{x:0.{prec}f}".format(x, prec)
    
    def chval(x, prec=1, *, add_par=''):  # change_value
        return f'style="padding-right:1.5ex;{add_par}"|—' if math.isnan(x) else \
               f'style="padding-right:1.5ex;color:darkgreen;{add_par}"|{x:0.{prec}f}' if x>0 else \
               f'style="padding-right:1.5ex;color:crimson;{add_par}"|−{-x:0.{prec}f}' if x<0 else \
               f'style="padding-right:1.5ex;color:darkgray;{add_par}"|{x:0.{prec}f}'
    
    def chval_bold(x, prec=1, *, add_par=''):  # change_value
        return f'style="padding-right:1.5ex;{add_par}"|\'\'\'—\'\'\'' if math.isnan(x) else \
               f'style="padding-right:1.5ex;color:darkgreen;{add_par}"|\'\'\'{x:0.{prec}f}\'\'\'' if x>0 else \
               f'style="padding-right:1.5ex;color:crimson;{add_par}"|\'\'\'−{-x:0.{prec}f}\'\'\'' if x<0 else \
               f'style="padding-right:1.5ex;color:darkgray;{add_par}"|\'\'\'{x:0.{prec}f}\'\'\''

    with open('design/' + file_header, mode='r', encoding="utf-8") as fh:
        table_header = fh.read()

    st = ''
    for i in range(len(df)):
        ser = df.iloc[i]
        if ser.name == 'ITALIA':
             st += '\n' + '|-class=static-row-header\n' + \
                  f'| \'\'\'{dd_replacement_provinces[ser.name][lang][0]}\'\'\' ' + \
                  f'||style="background:#e0ffd8;"| \'\'\'{if_value(ser["2014"])} ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{if_value(ser["m_2014"])} ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{if_value(ser["f_2014"])} ' + \
                  f'|| \'\'\'{if_value(ser["f-m_2014"])}\'\'\' ' + \
                  f'||{chval_bold(ser["2014→2019"], add_par="border-left-width:2px;")} ' + \
                  f'||style="border-left-width:2px;background:#e0ffd8;"| \'\'\'{if_value(ser["2019"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{if_value(ser["m_2019"])}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{if_value(ser["f_2019"])}\'\'\' ' + \
                  f'|| \'\'\'{if_value(ser["f-m_2019"])}\'\'\' ' + \
                  f'||{chval_bold(ser["2019→2024"], add_par="border-left-width:2px;")} ' + \
                  f'||style="border-left-width:2px;background:#e0ffd8;"| \'\'\'{if_value(ser["2024"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{if_value(ser["m_2024"])}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{if_value(ser["f_2024"])}\'\'\' ' + \
                  f'|| \'\'\'{if_value(ser["f-m_2024"])}\'\'\' ' + \
                  f'||{chval_bold(ser["2014→2024"], add_par="border-left-width:2px;")}'
        else:
            name_link = dd_replacement_provinces[ser.name][lang][1]
            name_visible = dd_replacement_provinces[ser.name][lang][0]
            name_inserted = name_link if name_link == name_visible else f"{name_link}|{name_visible}"
            st += '\n' + '|-\n' + \
                  f'| [[{name_inserted}]] ' + \
                  f'||style="background:#e0ffd8;"| {if_value(ser["2014"])} ' + \
                  f'||style="background:#eaf3ff;"| {if_value(ser["m_2014"])} ' + \
                  f'||style="background:#fee7f6;"| {if_value(ser["f_2014"])} ' + \
                  f'|| {if_value(ser["f-m_2014"])} ' + \
                  f'||{chval(ser["2014→2019"], add_par="border-left-width:2px;")} ' + \
                  f'||style="border-left-width:2px;background:#e0ffd8;"| {if_value(ser["2019"])} ' + \
                  f'||style="background:#eaf3ff;"| {if_value(ser["m_2019"])} ' + \
                  f'||style="background:#fee7f6;"| {if_value(ser["f_2019"])} ' + \
                  f'|| {if_value(ser["f-m_2019"])} ' + \
                  f'||{chval(ser["2019→2024"], add_par="border-left-width:2px;")} ' + \
                  f'||style="border-left-width:2px;background:#e0ffd8;"| {if_value(ser["2024"])} ' + \
                  f'||style="background:#eaf3ff;"| {if_value(ser["m_2024"])} ' + \
                  f'||style="background:#fee7f6;"| {if_value(ser["f_2024"])} ' + \
                  f'|| {if_value(ser["f-m_2024"])} ' + \
                  f'||{chval(ser["2014→2024"], add_par="border-left-width:2px;")}'

    if lang == 'ru':
        st = re.sub('(?<=\\d)\\.(?=\\d)', ',', st)  # replace . to comma, if this . is between two digits
        st = st.replace('padding-right:1,5ex;', 'padding-right:1.5ex;')

    st = table_header + st + '\n|}'
    
    # gray color for missing values
    st = st.replace(';"| —', ';color:silver;"| —')

    return st


table_code = create_table_provinces_v2(df_provinces, file_header='Italian_provinces_header_ru -2024 -v2.txt', lang='ru')
# write the code to file
with open('output/Table code for Italian provinces -ru -v2.txt', 'w', encoding="utf-8") as fh:
    fh.write(table_code)


table_code = create_table_provinces_v2(df_provinces, file_header='Italian_provinces_header_en -2024 -v2.txt', lang='en')
# write the code to file
with open('output/Table code for Italian provinces -en -v2.txt', 'w', encoding="utf-8") as fh:
    fh.write(table_code)